In [ ]:
!git clone https://github.com/sidms24/group-b
%cd group-b
!pip install -q -r requirements.txt

In [ ]:
import sys, warnings
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from sections.ppo_sac import (
    run_ppo_experiment, run_ppo_selfplay_experiment,
    run_sac_experiment,
    evaluate_ppo, make_bid_fn_ppo, export_policy_weights,
    raw_state_features, binned_state_features, make_tile_state_features,
    N_STATE_RAW, N_STATE_BINNED,
    PolicyNetwork,
)
from sections.distributional_nn import (
    round_robin_tournament, make_bid_fn_heuristic,
)

SEED = 24
print(f'JAX devices: {jax.devices()}')

In [ ]:
# Shared hyperparameters
N = 200_000
N_ACT = 51

tile_state_fn, N_STATE_TILE = make_tile_state_features(n_tilings=4, n_tiles=8)
print(f'State feature dims: raw={N_STATE_RAW}, binned={N_STATE_BINNED}, tile={N_STATE_TILE}')

## PPO-SAC: Standard Training vs Default Heuristic

Train PPO with SAC-style auto entropy tuning across different feature representations.

In [ ]:
print('PPO-SAC - Raw features (4-dim)...')
ppo_raw = run_ppo_experiment(
    N=N, eval_every=1000, n_actions=N_ACT,
    lr_actor=3e-4, lr_critic=1e-3, lr_alpha=3e-4,
    hidden=256, n_epochs=4, batch_size=2048, n_eval=2000,
    seed=SEED, state_fn=raw_state_features, n_state_features=N_STATE_RAW,
    clip_ratio=0.2, gae_lambda=0.95,
)
print(f'PPO-SAC Raw: best reward = {ppo_raw["best_reward"]:.2f}')

In [ ]:
print('PPO-SAC - Binned features (41-dim)...')
ppo_binned = run_ppo_experiment(
    N=N, eval_every=1000, n_actions=N_ACT,
    lr_actor=3e-4, lr_critic=1e-3, lr_alpha=3e-4,
    hidden=256, n_epochs=4, batch_size=2048, n_eval=2000,
    seed=SEED, state_fn=binned_state_features, n_state_features=N_STATE_BINNED,
    clip_ratio=0.2, gae_lambda=0.95,
)
print(f'PPO-SAC Binned: best reward = {ppo_binned["best_reward"]:.2f}')

In [ ]:
print('PPO-SAC - Tile features (512-dim)...')
ppo_tile = run_ppo_experiment(
    N=N, eval_every=1000, n_actions=N_ACT,
    lr_actor=3e-4, lr_critic=1e-3, lr_alpha=3e-4,
    hidden=256, n_epochs=4, batch_size=2048, n_eval=2000,
    seed=SEED, state_fn=tile_state_fn, n_state_features=N_STATE_TILE,
    clip_ratio=0.2, gae_lambda=0.95,
)
print(f'PPO-SAC Tile: best reward = {ppo_tile["best_reward"]:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(N // 1000) * 1000

# Rewards
axes[0].plot(x, ppo_raw['rewards'], label='Raw (4-dim)', alpha=0.8)
axes[0].plot(x, ppo_binned['rewards'], label='Binned (41-dim)', alpha=0.8)
axes[0].plot(x, ppo_tile['rewards'], label='Tile (512-dim)', alpha=0.8)
axes[0].set_xlabel('Episodes'); axes[0].set_ylabel('Greedy Reward')
axes[0].set_title('PPO-SAC: Feature Comparison'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Entropy
axes[1].plot(x, ppo_raw['entropy'], label='Raw', alpha=0.8)
axes[1].plot(x, ppo_binned['entropy'], label='Binned', alpha=0.8)
axes[1].plot(x, ppo_tile['entropy'], label='Tile', alpha=0.8)
axes[1].set_xlabel('Episodes'); axes[1].set_ylabel('Policy Entropy')
axes[1].set_title('Entropy (auto-tuned)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Alpha
axes[2].plot(x, ppo_raw['alpha'], label='Raw', alpha=0.8)
axes[2].plot(x, ppo_binned['alpha'], label='Binned', alpha=0.8)
axes[2].plot(x, ppo_tile['alpha'], label='Tile', alpha=0.8)
axes[2].set_xlabel('Episodes'); axes[2].set_ylabel('Alpha')
axes[2].set_title('Temperature Alpha'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('PPO-SAC: Standard Training', fontsize=14)
plt.tight_layout(); plt.show()

## PPO-SAC Self-Play Training

Train via self-play against a frozen copy of the agent. The opponent is updated periodically.

In [ ]:
# Pick best feature representation
ppo_results = {
    'Raw': (ppo_raw, raw_state_features, N_STATE_RAW),
    'Binned': (ppo_binned, binned_state_features, N_STATE_BINNED),
    'Tile': (ppo_tile, tile_state_fn, N_STATE_TILE),
}
best_name = max(ppo_results, key=lambda k: ppo_results[k][0]['best_reward'])
_, best_sfn, best_nsf = ppo_results[best_name]
print(f'Best feature repr: {best_name} (reward={ppo_results[best_name][0]["best_reward"]:.2f})')
print(f'Using {best_name} features for self-play and SAC experiments.')

In [ ]:
print('PPO-SAC Self-Play...')
ppo_sp = run_ppo_selfplay_experiment(
    N=N, eval_every=1000, n_actions=N_ACT,
    lr_actor=3e-4, lr_critic=1e-3, lr_alpha=3e-4,
    hidden=256, n_epochs=4, batch_size=2048, n_eval=2000,
    seed=SEED, state_fn=best_sfn, n_state_features=best_nsf,
    clip_ratio=0.2, gae_lambda=0.95,
    opponent_update_every=10,
)
print(f'PPO-SAC Self-Play: best reward = {ppo_sp["best_reward"]:.2f}')

In [ ]:
# Self-play with higher entropy for more exploration
print('PPO-SAC Self-Play (high entropy)...')
ppo_sp_high_ent = run_ppo_selfplay_experiment(
    N=N, eval_every=1000, n_actions=N_ACT,
    lr_actor=3e-4, lr_critic=1e-3, lr_alpha=1e-4,
    hidden=256, n_epochs=4, batch_size=2048, n_eval=2000,
    seed=SEED, state_fn=best_sfn, n_state_features=best_nsf,
    clip_ratio=0.2, gae_lambda=0.95,
    target_entropy=-0.3 * np.log(N_ACT),  # higher target entropy
    opponent_update_every=5,  # faster opponent updates
)
print(f'PPO-SAC Self-Play (high ent): best reward = {ppo_sp_high_ent["best_reward"]:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(N // 1000) * 1000

best_std = ppo_results[best_name][0]
axes[0].plot(x, best_std['rewards'], label=f'PPO-SAC {best_name} (vs heuristic)', alpha=0.8)
axes[0].plot(x, ppo_sp['rewards'], label='PPO-SAC Self-Play', alpha=0.8)
axes[0].plot(x, ppo_sp_high_ent['rewards'], label='PPO-SAC Self-Play (high ent)', alpha=0.8)
axes[0].set_xlabel('Episodes'); axes[0].set_ylabel('Greedy Reward (vs heuristic)')
axes[0].set_title('PPO-SAC: Standard vs Self-Play')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(x, ppo_sp['selfplay_rewards'], label='Self-play return', alpha=0.8)
axes[1].plot(x, ppo_sp_high_ent['selfplay_rewards'], label='Self-play return (high ent)', alpha=0.8)
axes[1].set_xlabel('Episodes'); axes[1].set_ylabel('Mean Episode Return')
axes[1].set_title('Self-Play Training Return')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Discrete SAC

Off-policy SAC with replay buffer, twin Q-networks, and auto entropy temperature.

In [ ]:
print('Discrete SAC...')
sac_result = run_sac_experiment(
    N=N, eval_every=1000, n_actions=N_ACT,
    lr_actor=3e-4, lr_q=3e-4, lr_alpha=3e-4,
    hidden=256, batch_size=256, buffer_size=100_000,
    n_eval=2000, seed=SEED,
    state_fn=best_sfn, n_state_features=best_nsf,
    n_updates_per_chunk=10, warmup_episodes=2000,
)
print(f'SAC: best reward = {sac_result["best_reward"]:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(N // 1000) * 1000

axes[0].plot(x, best_std['rewards'], label=f'PPO-SAC {best_name}', alpha=0.8)
axes[0].plot(x, ppo_sp['rewards'], label='PPO-SAC Self-Play', alpha=0.8)
axes[0].plot(x, sac_result['rewards'], label='Discrete SAC', alpha=0.8)
axes[0].set_xlabel('Episodes'); axes[0].set_ylabel('Greedy Reward')
axes[0].set_title('PPO-SAC vs SAC'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(x, sac_result['entropy'], label='SAC entropy', alpha=0.8)
axes[1].plot(x, ppo_sp['entropy'], label='PPO-SAC entropy', alpha=0.8)
axes[1].set_xlabel('Episodes'); axes[1].set_ylabel('Policy Entropy')
axes[1].set_title('Entropy Comparison'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## Round-Robin Tournament

Compare all PPO-SAC and SAC variants head-to-head against each other and the heuristic.

In [ ]:
agents = {}

# PPO-SAC variants
ppo_variants = {
    'PPO Raw':        (ppo_raw, raw_state_features),
    'PPO Binned':     (ppo_binned, binned_state_features),
    'PPO Tile':       (ppo_tile, tile_state_fn),
    'PPO Self-Play':  (ppo_sp, best_sfn),
    'PPO SP High-Ent':(ppo_sp_high_ent, best_sfn),
    'SAC':            (sac_result, best_sfn),
}

for name, (res, sfn) in ppo_variants.items():
    agents[name] = make_bid_fn_ppo(res['actor_params_best'], sfn, n_actions=N_ACT, hidden=256)

agents['Heuristic'] = make_bid_fn_heuristic()

print(f'Tournament agents ({len(agents)}):')
for name in agents:
    print(f'  - {name}')

In [ ]:
print('Running round-robin tournament...')
tournament = round_robin_tournament(agents, n_games=1000, seed=SEED)

print(f'\nTournament ranking:')
print(f'{"Rank":>4s}  {"Agent":30s}  {"Avg Win Rate":>12s}  {"Avg Reward":>12s}')
print('-' * 62)
for rank, (name, wr) in enumerate(tournament['ranking'], 1):
    idx = tournament['names'].index(name)
    mr = tournament['total_reward'][idx]
    print(f'{rank:4d}  {name:30s}  {wr:12.3f}  {mr:12.1f}')

In [ ]:
# Win-rate heatmap
names = tournament['names']
n = len(names)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
im = ax.imshow(tournament['win_matrix'], cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(n)); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(n)); ax.set_yticklabels(names, fontsize=7)
ax.set_title('Win Rate Matrix (row beats column)')
for i in range(n):
    for j in range(n):
        if i != j:
            val = tournament['win_matrix'][i, j]
            color = 'white' if val > 0.7 or val < 0.3 else 'black'
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=6, color=color)
plt.colorbar(im, ax=ax, label='Win Rate')

ax = axes[1]
im2 = ax.imshow(tournament['reward_matrix'], cmap='viridis', aspect='auto')
ax.set_xticks(range(n)); ax.set_xticklabels(names, rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(n)); ax.set_yticklabels(names, fontsize=7)
ax.set_title('Mean Reward Matrix (row vs column)')
for i in range(n):
    for j in range(n):
        if i != j:
            ax.text(j, i, f'{tournament["reward_matrix"][i,j]:.0f}',
                    ha='center', va='center', fontsize=6, color='white')
plt.colorbar(im2, ax=ax, label='Mean Reward')

plt.suptitle('PPO-SAC Tournament Results', fontsize=14)
plt.tight_layout(); plt.show()

## Summary and Export Best Agent

In [ ]:
# Summary table
summary = {}
for name, (res, _) in ppo_variants.items():
    summary[name] = res['best_reward']

print(f'{"Method":30s} {"Best Greedy Reward":>20s}')
print('-' * 52)
for name, rew in sorted(summary.items(), key=lambda x: -x[1]):
    marker = ' << BEST' if rew == max(summary.values()) else ''
    print(f'{name:30s} {rew:20.2f}{marker}')

fig, ax = plt.subplots(figsize=(10, 5))
names_s = list(sorted(summary.keys(), key=lambda k: summary[k]))
values_s = [summary[n] for n in names_s]
colors = ['#4CAF50' if v == max(values_s) else '#90CAF9' for v in values_s]
ax.barh(range(len(names_s)), values_s, color=colors)
ax.set_yticks(range(len(names_s)))
ax.set_yticklabels(names_s, fontsize=9)
ax.set_xlabel('Best Greedy Reward (vs Heuristic)')
ax.set_title('PPO-SAC Method Comparison')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Pick the best tournament performer for export
best_tournament_name = tournament['ranking'][0][0]
print(f'Best tournament agent: {best_tournament_name}')

best_res, best_export_sfn = ppo_variants[best_tournament_name]
best_params = best_res['actor_params_best']

# Determine feature type from the variant name
if best_export_sfn is raw_state_features:
    feat_type = 'raw'
elif best_export_sfn is binned_state_features:
    feat_type = 'binned'
else:
    feat_type = 'tile'
print(f'Feature type: {feat_type}')


export_policy_weights(best_params, 'said_weight/ppo_sac_weights.npz', feature_type=feat_type)
